# Fase 1 — Ingestão Tabular 1:1 das Tabelas Eleitorais (Eleições 2026)

Visão geral da Fase 1: Ingestão tabular 1:1 das 17 tabelas e dimensões das Eleições 2026 do TSE em formato Parquet ZSTD particionado (`election_year=2026/turn=T/uf=UF`).

### Objetivos:
- Download e extração resiliente de dados oficiais do TSE
- Normalização e validação de schema canônico para 17 tabelas eleitorais
- Particionamento e persistência colunar em Parquet comprimido com ZSTD
- Garantia de idempotência e checkpointing incremental por estado (UF) e turno

In [ ]:
from pathlib import Path

# Parâmetros idempotentes de execução
ELECTION_YEAR = 2026
TURNS = [1, 2]
TARGET_UFS = ["AC"]
SKIP_EXISTING = True
FORCE_REFRESH = False
STAGING_DIR = Path("data/staging")
EXPORTS_DIR = Path("data/exports")
MANIFEST_PATH = Path("data/exports/_checkpoints/manifest.json")

# Catálogo declarativo das 17 tabelas eleitorais do TSE
ELECTORAL_TABLES_CATALOG = {
    "fact_boletim_urna": {
        "description": "Boletim de Urna (BU) consolidado por seção",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_votacao_secao": {
        "description": "Votação nominal e de legenda por seção eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_detalhe_votacao_secao": {
        "description": "Detalhe de apuração da votação por seção eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_detalhe_votacao_munzona": {
        "description": "Detalhe da apuração da votação por município e zona eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "dim_candidato_munzona": {
        "description": "Dimensão de candidatos por município e zona eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "dim_partido_munzona": {
        "description": "Dimensão de partidos/coligações por município e zona eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_eleitorado_perfil_secao": {
        "description": "Perfil do eleitorado (faixa etária, gênero, escolaridade) por seção",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_eleitorado_perfil_munzona": {
        "description": "Perfil do eleitorado agregado por município e zona eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "dim_local_votacao_secao": {
        "description": "Local físico de votação vinculado a cada seção eleitoral",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_eleitorado_deficiencia": {
        "description": "Eleitores com deficiência declarada por local e seção",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_transferencia_temporaria_secao": {
        "description": "Transferências temporárias de eleitores (TTE) por seção",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_transferencia_temporaria_mun": {
        "description": "Transferências temporárias de eleitores (TTE) por município",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_comparecimento_abstencao": {
        "description": "Estatísticas de comparecimento e abstenção por seção",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_comparecimento_abstencao_pcd": {
        "description": "Comparecimento e abstenção de pessoas com deficiência",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "fact_comparecimento_abstencao_tte": {
        "description": "Comparecimento e abstenção de eleitores em transferência temporária",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "bridge_correspondencia_esperada": {
        "description": "Tabela bridge de correspondência esperada de urnas eletrônicas",
        "partition_cols": ["election_year", "turn", "uf"],
    },
    "bridge_correspondencia_efetivada": {
        "description": "Tabela bridge de correspondência efetivada de urnas eletrônicas",
        "partition_cols": ["election_year", "turn", "uf"],
    },
}
